### Indicadores do OnePage

- Faturamento -> Meta Ano: 1.650.000 / Meta Dia: 1000
- Diversidade de Produtos (quantos produtos diferentes foram vendidos naquele período) -> Meta Ano: 120 / Meta Dia: 4
- Ticket Médio por Venda -> Meta Ano: 500 / Meta Dia: 500

Obs: Cada indicador deve ser calculado no dia e no ano. O indicador do dia deve ser o do último dia disponível na planilha de Vendas (a data mais recente)


In [104]:
import pandas as pd
from pathlib import Path
import sys
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from email.mime.application import MIMEApplication
from email import encoders
sys.path.append(r"C:\Users\Nathan\OneDrive\Documentos\Cursos\Comunidade Impressionadora\Python Impressionador\Intermediário\Integração Python - E-mail\SMTP\anexos")
from senha import senha_app
cwd = Path.cwd()
bases_de_dados = Path(f"{cwd}/Bases de Dados").iterdir()
backup = Path(f"{cwd}/Backup Arquivos Lojas")


meta_ano = 1650000
meta_dia = 1000
meta_prod_ano = 120
meta_prod_dia = 4
meta_ticket_ano = 500
meta_ticket_dia = 500

df_email = pd.read_excel(f"{cwd}/Bases de Dados/Emails.xlsx")
df_lojas = pd.read_csv(f"{cwd}/Bases de Dados/Lojas.csv",encoding="latin1",sep=";",index_col="ID Loja")
df_vendas = pd.read_excel(f"{cwd}/Bases de Dados/Vendas.xlsx")
df_geral = df_vendas.merge(df_lojas.reset_index(), on="ID Loja")
df_geral = df_geral.merge(df_email, on="Loja")


display(df_geral)
display(df_email)

,Código Venda,Data,ID Loja,Produto,Quantidade,Valor Unitário,Valor Final,Loja,Gerente,E-mail
0,1,2019-01-01,1,Sapato Estampa,1,358,358,Iguatemi Esplanada,Helena,nathanps337@gmail.com
1,1,2019-01-01,1,Camiseta,2,180,360,Iguatemi Esplanada,Helena,nathanps337@gmail.com
2,1,2019-01-01,1,Sapato Xadrez,1,368,368,Iguatemi Esplanada,Helena,nathanps337@gmail.com
3,2,2019-01-02,3,Relógio,3,200,600,Norte Shopping,Laura,nathanps337@gmail.com
4,2,2019-01-02,3,Chinelo Liso,1,71,71,Norte Shopping,Laura,nathanps337@gmail.com
...,...,...,...,...,...,...,...,...,...,...
100994,69996,2019-12-26,17,Short Listrado,2,102,204,Center Shopping Uberlândia,Arthur,nathansousa2n@gmail.com
100995,69996,2019-12-26,17,Mochila,4,270,1080,Center Shopping Uberlândia,Arthur,nathansousa2n@gmail.com
100996,69996,2019-12-26,17,Pulseira Estampa,1,87,87,Center Shopping Uberlândia,Arthur,nathansousa2n@gmail.com
100997,69997,2019-12-26,11,Camisa Listrado,1,108,108,Ribeirão Shopping,Lorena,nathansousa2n@gmail.com


,Loja,Gerente,E-mail
0,Iguatemi Esplanada,Helena,nathanps337@gmail.com
1,Shopping Midway Mall,Alice,nathanps337@gmail.com
2,Norte Shopping,Laura,nathanps337@gmail.com
3,Shopping Iguatemi Fortaleza,Manuela,nathanps337@gmail.com
4,Shopping União de Osasco,Valentina,nathanps337@gmail.com
5,Shopping Center Interlagos,Sophia,nathanps337@gmail.com
6,Rio Mar Recife,Isabella,nathanps337@gmail.com
7,Salvador Shopping,Heloisa,nathanps337@gmail.com
8,Rio Mar Shopping Fortaleza,Luiza,nathansousa2n@gmail.com
9,Shopping Center Leste Aricanduva,Julia,nathansousa2n@gmail.com


META ANUAL

In [105]:
ranking_ano = df_geral.groupby("Loja")[["Valor Final"]].sum().sort_values(ascending=False, by="Valor Final")
ranking_ano["Meta"] = None  
for i in ranking_ano.index:
    if ranking_ano.loc[i, "Valor Final"] < meta_ano:
        ranking_ano.loc[i, "Meta"]="Não Bateu"
    else:
        ranking_ano.loc[i, "Meta"]="Bateu"

display(ranking_ano)
pior_loja_ano = ranking_ano["Valor Final"].idxmin()
melhor_loja_ano = ranking_ano["Valor Final"].idxmax()
print(melhor_loja_ano)
print(pior_loja_ano)



,Valor Final,Meta
Loja,,
Iguatemi Campinas,1762419,Bateu
Shopping Vila Velha,1731167,Bateu
Bourbon Shopping SP,1726110,Bateu
Rio Mar Recife,1722766,Bateu
Shopping SP Market,1721763,Bateu
Palladium Shopping Curitiba,1721120,Bateu
Norte Shopping,1711968,Bateu
Ribeirão Shopping,1707122,Bateu
Iguatemi Esplanada,1699681,Bateu


Iguatemi Campinas
Shopping Morumbi


META DIÁRIA

In [106]:
data = df_geral.loc[df_geral["Data"].idxmax(), "Data"].date()

ranking_dia = df_geral[df_geral["Data"].dt.date==data]
# display(df_geral['Loja'].unique())
# display(ranking_dia["Loja"].unique())
ranking_dia = ranking_dia.groupby("Loja")[["Valor Final"]].sum().sort_values(ascending=False, by="Valor Final")
ranking_dia["Meta"] = None
for i in ranking_dia.index:
    if ranking_dia.loc[i,"Valor Final"]<meta_dia:
        ranking_dia.loc[i,"Meta"] = "Não Bateu"
    else:
        ranking_dia.loc[i,"Meta"] = "Bateu"
display(ranking_dia)
pior_loja_dia = ranking_dia["Valor Final"].idxmin()
melhor_loja_dia = ranking_dia["Valor Final"].idxmax()
print(melhor_loja_dia)
print(pior_loja_dia)


,Valor Final,Meta
Loja,,
Salvador Shopping,3950,Bateu
Novo Shopping Ribeirão Preto,3400,Bateu
Center Shopping Uberlândia,2651,Bateu
Shopping Eldorado,2391,Bateu
Shopping Center Interlagos,1582,Bateu
Shopping Recife,1366,Bateu
Norte Shopping,1259,Bateu
Shopping União de Osasco,1207,Bateu
Shopping Vila Velha,937,Não Bateu


Salvador Shopping
Shopping Ibirapuera


DIVERSIDADE DE PRODUTOS (ANO)

In [107]:
diversidade_ano = df_geral.groupby("Loja")[["Produto"]].nunique()
display(diversidade_ano)

,Produto
Loja,
Bourbon Shopping SP,120
Center Shopping Uberlândia,120
Iguatemi Campinas,120
Iguatemi Esplanada,120
Norte Shopping,120
Novo Shopping Ribeirão Preto,120
Palladium Shopping Curitiba,120
Parque Dom Pedro Shopping,120
Passei das Águas Shopping,120


DIVERSIDADE DE PRODUTOS (DIA)

In [108]:
diversidade_dia = df_geral[df_geral['Data'].dt.date==data]
diversidade_dia = diversidade_dia.groupby("Loja")[["Produto"]].nunique().sort_values(ascending=False, by="Produto")
diversidade_dia = diversidade_dia
diversidade_dia["Meta"] =  None
for i in diversidade_dia.index:
    if diversidade_dia.loc[i, "Produto"] < meta_prod_dia:
        diversidade_dia.loc[i,"Meta"] = "Não Bateu"
    else:
        diversidade_dia.loc[i,"Meta"]= "Bateu"


display(diversidade_dia)
pior_loja_dia = diversidade_dia["Produto"].idxmin()
melhor_loja_dia = diversidade_dia["Produto"].idxmax()
print(melhor_loja_dia)
print(pior_loja_dia)


,Produto,Meta
Loja,,
Novo Shopping Ribeirão Preto,6,Bateu
Center Shopping Uberlândia,4,Bateu
Ribeirão Shopping,3,Não Bateu
Shopping União de Osasco,3,Não Bateu
Shopping Vila Velha,3,Não Bateu
Shopping Recife,3,Não Bateu
Salvador Shopping,2,Não Bateu
Norte Shopping,2,Não Bateu
Shopping Center Leste Aricanduva,2,Não Bateu


Novo Shopping Ribeirão Preto
Rio Mar Shopping Fortaleza


TICKET MÉDIO POR VENDA (ANO)

In [109]:

ticket_ano = df_geral.groupby(["Loja", "Código Venda"])[["Valor Final"]].sum(numeric_only=True).groupby("Loja").mean().sort_values(ascending=False, by="Valor Final")

display(ticket_ano)
pior_loja_dia = ticket_ano["Valor Final"].idxmin()
melhor_loja_dia = ticket_ano["Valor Final"].idxmax()
print(melhor_loja_dia)
print(pior_loja_dia)

,Valor Final
Loja,
Iguatemi Campinas,822.023787
Ribeirão Shopping,806.006610
Shopping SP Market,800.447699
Shopping União de Osasco,796.062201
Iguatemi Esplanada,795.358446
Rio Mar Recife,790.259633
Shopping Recife,785.345094
Novo Shopping Ribeirão Preto,785.318203
Norte Shopping,784.586618


Iguatemi Campinas
Shopping Ibirapuera


TICKET MEDIO (DIA)

In [110]:
ticket_dia = df_geral[df_geral['Data'].dt.date==data]
ticket_dia = ticket_dia.groupby(["Loja", "Código Venda"])[["Valor Final"]].sum(numeric_only=True).groupby("Loja").mean().sort_values(ascending=False, by="Valor Final")
display(ticket_dia)
pior_loja_dia = ticket_dia["Valor Final"].idxmin()
melhor_loja_dia = ticket_dia["Valor Final"].idxmax()
print(melhor_loja_dia)
print(pior_loja_dia)

,Valor Final
Loja,
Salvador Shopping,3950.000000
Shopping Eldorado,2391.000000
Novo Shopping Ribeirão Preto,1700.000000
Shopping Center Interlagos,1582.000000
Center Shopping Uberlândia,1325.500000
Norte Shopping,1259.000000
Shopping União de Osasco,1207.000000
Shopping Recife,683.000000
Bourbon Shopping SP,676.000000


Salvador Shopping
Shopping Ibirapuera


PREPARANDO OS EMAILS


In [111]:
servidor = smtplib.SMTP('smtp.gmail.com', 587)
servidor.starttls() # Inicia a criptografia
servidor.login("nathansousa2n@gmail.com", senha_app) # Faz o login
for i in df_email.index:
    nome_gerente = df_email.loc[i, "Gerente"] # Ajuste para o nome da sua coluna
    nome_loja = df_email.loc[i, "Loja"]
    #criando backups e arquivos
    pasta_loja = Path(f"Backup Arquivos Lojas/{nome_loja}")
    pasta_loja.mkdir(parents=True, exist_ok=True)
    nome_arquivo = f"{nome_loja}-{data.strftime('%d-%m-%Y')}.xlsx"
    caminho_completo = pasta_loja/nome_arquivo
    #Diretoria
    pasta_diretoria = Path(f"Backup Arquivos Lojas/Diretoria")
    pasta_diretoria.mkdir(parents=True, exist_ok=True)
    # Arquivo do Ano
    nome_arquivo_ano = f"Ranking_Ano-{data.strftime('%d-%m-%Y')}.xlsx"
    caminho_diretoria_ano = pasta_diretoria / nome_arquivo_ano
    
    # Arquivo do Dia
    nome_arquivo_dia = f"Ranking_Dia-{data.strftime('%d-%m-%Y')}.xlsx"
    caminho_diretoria_dia = pasta_diretoria / nome_arquivo_dia

    if nome_loja not in ranking_dia.index and nome_gerente != "Diretoria":
        continue
    if nome_gerente != "Diretoria":
        df_resultados = df_geral[df_geral["Loja"]==nome_loja]
        df_resultados.to_excel(caminho_completo, index=False)
        resultado_dia = ranking_dia.loc[nome_loja, "Valor Final"]
        resultado_ano = ranking_ano.loc[nome_loja, "Valor Final"]
        r_diversidade_dia = diversidade_dia.loc[nome_loja, "Produto"]
        r_diversidade_ano = diversidade_ano.loc[nome_loja, "Produto"]
        r_ticket_dia = ticket_dia.loc[nome_loja, "Valor Final"]
        r_ticket_ano = ticket_ano.loc[nome_loja, "Valor Final"]

    msg = MIMEMultipart()
    msg["Subject"] = f"One Page Dia {data.strftime('%d/%m/%Y')} - Loja {nome_loja}"
    msg["From"] = "nathansousa2n@gmail.com"
    msg["To"] = df_email.loc[i, "E-mail"]
    if nome_gerente != "Diretoria":
        corpo_email = f""" 
<html>
<head>
    <style>
        body {{ font-family: Arial, sans-serif; color: #333333; line-height: 1.6; }}
        table {{ border-collapse: collapse; width: 100%; max-width: 500px; margin-top: 10px; margin-bottom: 20px; }}
        th, td {{ border: 1px solid #dddddd; text-align: center; padding: 8px; font-size: 14px; }}
        th {{ background-color: #f2f2f2; font-weight: bold; }}
    </style>
</head>
<body>
    <p>Bom dia, <strong>{nome_gerente}</strong></p>
    
    <p>O resultado de ontem ({data.strftime('%d/%m/%Y')}) da <strong>{nome_loja}</strong> foi:</p>
    
    <table>
        <thead>
            <tr>
                <th>Indicador</th>
                <th>Valor Dia</th>
                <th>Meta Dia</th>
                <th>Cenário Dia</th>
            </tr>
        </thead>
        <tbody>
            <tr>
                <td>Faturamento</td>
                <td>{resultado_dia}</td>
                <td>{meta_dia}</td>
                <td>{"✅" if resultado_dia >= meta_dia else "❌"}</td>
            </tr>
            <tr>
                <td>Diversidade de Produtos</td>
                <td>{r_diversidade_dia}</td>
                <td>{meta_prod_dia}</td>
                <td>{"✅" if r_diversidade_dia >= meta_prod_dia else "❌"}</td>
            </tr>
            <tr>
                <td>Ticket Médio</td>
                <td>{r_ticket_dia}</td>
                <td>{meta_ticket_dia}</td>
                <td>{"✅" if r_ticket_dia >= meta_ticket_dia else "❌"}</td>
            </tr>
        </tbody>
    </table>

    <table>
        <thead>
            <tr>
                <th>Indicador</th>
                <th>Valor Ano</th>
                <th>Meta Ano</th>
                <th>Cenário Ano</th>
            </tr>
        </thead>
        <tbody>
            <tr>
                <td>Faturamento</td>
                <td>{resultado_ano}</td>
                <td>{meta_ano}</td>
                <td>{"✅" if resultado_ano >= meta_ano else "❌"}</td>
            </tr>
            <tr>
                <td>Diversidade de Produtos</td>
                <td>{r_diversidade_ano}</td>
                <td>{meta_prod_ano}</td>
                <td>{"✅" if r_diversidade_ano>= meta_prod_ano else "❌"}</td>
            </tr>
            <tr>
                <td>Ticket Médio</td>
                <td>{r_ticket_ano}</td>
                <td>{meta_ticket_ano}</td>
                <td>{"✅" if r_ticket_ano >= meta_ticket_ano else "❌"}</td>
            </tr>
        </tbody>
    </table>

    <p>Segue em anexo a planilha com todos os dados para mais detalhes.<br>
    Qualquer dúvida, estou à disposição.</p>

</body>
</html>
"""
        msg.attach(MIMEText(corpo_email, 'html', 'utf-8'))
        # msg.
        with open(caminho_completo, "rb") as arquivo:
         msg.attach(MIMEApplication(arquivo.read(), Name=nome_arquivo))
    
    
    else:
        ranking_ano.to_excel(caminho_diretoria_ano, index=False)
        ranking_dia.to_excel(caminho_diretoria_dia, index=False)
        msg["Subject"] = f"One Page Dia {data.strftime('%d/%m/%Y')} - Dretoria"
        corpo_email_diretoria = f"""
<html>
<head>
    <style>
        body {{ font-family: Arial, sans-serif; color: #333333; line-height: 1.6; }}
        table {{ border-collapse: collapse; width: 100%; max-width: 500px; margin-top: 10px; margin-bottom: 20px; }}
        th, td {{ border: 1px solid #dddddd; text-align: center; padding: 8px; font-size: 14px; }}
        th {{ background-color: #f2f2f2; font-weight: bold; }}
    </style>
</head>
<body>
    <p>Bom dia,</p>

    <p>Segue o resumo de desempenho das lojas do dia {data.strftime('%d/%m/%Y')}:</p>

    <p>🏆 <strong>Melhor loja do dia:</strong> {ranking_dia["Valor Final"].idxmax()}</p>
    <p>📉 <strong>Pior loja do dia:</strong> {ranking_dia["Valor Final"].idxmin()}</p>

    <p>🏆 <strong>Melhor loja do ano:</strong> {ranking_ano["Valor Final"].idxmax()}</p>
    <p>📉 <strong>Pior loja do ano:</strong> {ranking_ano["Valor Final"].idxmin()}</p>

    <p>Seguem em anexo a planilha com todos os dados para mais detalhes.</p>
    <p>Qualquer dúvida, estou à disposição.</p>
</body>
</html>
"""
        msg.attach(MIMEText(corpo_email_diretoria, 'html', 'utf-8'))
# Anexando o Ranking do Ano
        with open(caminho_diretoria_ano, "rb") as arquivo_ano:
            msg.attach(MIMEApplication(arquivo_ano.read(), Name=nome_arquivo_ano))
            
        # Anexando o Ranking do Dia
        with open(caminho_diretoria_dia, "rb") as arquivo_dia:
            msg.attach(MIMEApplication(arquivo_dia.read(), Name=nome_arquivo_dia))

#fechando servidor
servidor.quit()
print("Automação finalizada!")

Automação finalizada!
